In [ ]:
# ============================================================
# PROJECT 5: TITANIC SURVIVAL
# FULL LOGISTIC REGRESSION PIPELINE
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

# ------------------------------------------------------------
# 2. LOAD DATASET
# ------------------------------------------------------------

from google.colab import files

uploaded = files.upload()

# Change filename if your file has a different name
df = pd.read_csv("Titanic-Dataset.csv")

print("Dataset loaded successfully!")

# ------------------------------------------------------------
# 3. BASIC DATA EXPLORATION
# ------------------------------------------------------------

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
display(df.describe())

# ------------------------------------------------------------
# 4. CHECK MISSING VALUES
# ------------------------------------------------------------

print("\nMissing Values:")
print(df.isnull().sum())

# ------------------------------------------------------------
# 5. CHECK DUPLICATES
# ------------------------------------------------------------

print("\nDuplicate Rows:")
print(df.duplicated().sum())

# ============================================================
# DATA PREPROCESSING
# ============================================================

# ------------------------------------------------------------
# 6. REMOVE UNNECESSARY COLUMNS
# ------------------------------------------------------------

df = df.drop(
    columns=[
        "PassengerId",
        "Name",
        "Ticket",
        "Cabin"
    ],
    errors="ignore"
)

print("\nColumns after removal:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 7. FEATURE ENGINEERING
# ------------------------------------------------------------

# Family size
df["FamilySize"] = (
    df["SibSp"] +
    df["Parch"] +
    1
)

# IsAlone feature
df["IsAlone"] = np.where(
    df["FamilySize"] == 1,
    1,
    0
)

print("\nFeature-engineered data:")
display(
    df[
        [
            "SibSp",
            "Parch",
            "FamilySize",
            "IsAlone"
        ]
    ].head()
)

# ------------------------------------------------------------
# 8. HANDLE MISSING VALUES
# ------------------------------------------------------------

# Age -> median
df["Age"] = df["Age"].fillna(
    df["Age"].median()
)

# Fare -> median
df["Fare"] = df["Fare"].fillna(
    df["Fare"].median()
)

# Embarked -> mode
df["Embarked"] = df["Embarked"].fillna(
    df["Embarked"].mode()[0]
)

print("\nMissing values after filling:")
print(df.isnull().sum())

# ------------------------------------------------------------
# 9. ENCODE SEX
# ------------------------------------------------------------

df["Sex"] = df["Sex"].map({
    "male": 0,
    "female": 1
})

# ------------------------------------------------------------
# 10. ONE-HOT ENCODE EMBARKED
# ------------------------------------------------------------

df = pd.get_dummies(
    df,
    columns=["Embarked"],
    drop_first=True,
    dtype=int
)

# ------------------------------------------------------------
# 11. CHECK FINAL DATA
# ------------------------------------------------------------

print("\nFinal Preprocessed Dataset:")
display(df.head())

print("\nFinal Shape:")
print(df.shape)

print("\nFinal Data Types:")
print(df.dtypes)

print("\nRemaining Missing Values:")
print(df.isnull().sum())

# ============================================================
# DEFINE FEATURES AND TARGET
# ============================================================

# ------------------------------------------------------------
# 12. X AND y
# ------------------------------------------------------------

X = df.drop(
    columns=["Survived"]
)

y = df["Survived"]

print("\nFeatures:")
display(X.head())

print("\nTarget:")
display(y.head())

# ------------------------------------------------------------
# 13. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Features Shape:")
print(X_train.shape)

print("\nTesting Features Shape:")
print(X_test.shape)

print("\nTraining Target Shape:")
print(y_train.shape)

print("\nTesting Target Shape:")
print(y_test.shape)

# ============================================================
# FEATURE SCALING
# ============================================================

# ------------------------------------------------------------
# 14. STANDARD SCALER
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

print("\nFeature scaling completed.")

# ============================================================
# LOGISTIC REGRESSION
# ============================================================

# ------------------------------------------------------------
# 15. CREATE MODEL
# ------------------------------------------------------------

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 16. TRAIN MODEL
# ------------------------------------------------------------

logistic_model.fit(
    X_train_scaled,
    y_train
)

print("\nLogistic Regression model trained successfully.")

# ------------------------------------------------------------
# 17. PREDICT CLASS
# ------------------------------------------------------------

y_pred = logistic_model.predict(
    X_test_scaled
)

# ------------------------------------------------------------
# 18. PREDICT PROBABILITY
# ------------------------------------------------------------

y_probability = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

# ============================================================
# MODEL EVALUATION
# ============================================================

# ------------------------------------------------------------
# 19. ACCURACY
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\nAccuracy:")
print(accuracy)

# ------------------------------------------------------------
# 20. PRECISION
# ------------------------------------------------------------

precision = precision_score(
    y_test,
    y_pred
)

print("\nPrecision:")
print(precision)

# ------------------------------------------------------------
# 21. RECALL
# ------------------------------------------------------------

recall = recall_score(
    y_test,
    y_pred
)

print("\nRecall:")
print(recall)

# ------------------------------------------------------------
# 22. F1 SCORE
# ------------------------------------------------------------

f1 = f1_score(
    y_test,
    y_pred
)

print("\nF1 Score:")
print(f1)

# ------------------------------------------------------------
# 23. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred
    )
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

# ------------------------------------------------------------
# 24. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)

# ------------------------------------------------------------
# 25. CONFUSION MATRIX VISUALIZATION
# ------------------------------------------------------------

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Did Not Survive",
        "Survived"
    ],
    yticklabels=[
        "Did Not Survive",
        "Survived"
    ]
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

# ============================================================
# ROC CURVE
# ============================================================

# ------------------------------------------------------------
# 26. ROC AUC SCORE
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("\nROC-AUC Score:")
print(roc_auc)

# ------------------------------------------------------------
# 27. ROC CURVE
# ------------------------------------------------------------

fpr, tpr, thresholds = roc_curve(
    y_test,
    y_probability
)

plt.figure(figsize=(7, 5))

plt.plot(
    fpr,
    tpr,
    label=f"ROC-AUC = {roc_auc:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.legend()

plt.show()

# ============================================================
# MODEL COEFFICIENTS
# ============================================================

# ------------------------------------------------------------
# 28. FEATURE COEFFICIENTS
# ------------------------------------------------------------

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": logistic_model.coef_[0]
})

coefficients["Absolute_Coefficient"] = (
    coefficients["Coefficient"].abs()
)

coefficients = coefficients.sort_values(
    by="Absolute_Coefficient",
    ascending=False
)

print("\nLogistic Regression Coefficients:")

display(coefficients)

# ------------------------------------------------------------
# 29. COEFFICIENT VISUALIZATION
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.barh(
    coefficients["Feature"],
    coefficients["Coefficient"]
)

plt.title(
    "Logistic Regression Feature Coefficients"
)

plt.xlabel("Coefficient")
plt.ylabel("Feature")

plt.gca().invert_yaxis()

plt.show()

# ============================================================
# ACTUAL VS PREDICTED
# ============================================================

# ------------------------------------------------------------
# 30. CREATE PREDICTION TABLE
# ------------------------------------------------------------

prediction_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Survival_Probability": y_probability
})

print("\nPrediction Results:")

display(
    prediction_df.head(20)
)

# ============================================================
# SAMPLE PASSENGER PREDICTION
# ============================================================

# ------------------------------------------------------------
# 31. CREATE A SAMPLE PASSENGER
# ------------------------------------------------------------

sample_passenger = pd.DataFrame({
    "Pclass": [3],
    "Sex": [0],
    "Age": [25],
    "SibSp": [0],
    "Parch": [0],
    "Fare": [10],
    "FamilySize": [1],
    "IsAlone": [1],
    "Embarked_Q": [0],
    "Embarked_S": [1]
})

# Make sure columns are in exactly the same order
sample_passenger = sample_passenger[
    X.columns
]

# ------------------------------------------------------------
# 32. SCALE SAMPLE
# ------------------------------------------------------------

sample_scaled = scaler.transform(
    sample_passenger
)

# ------------------------------------------------------------
# 33. PREDICT SAMPLE
# ------------------------------------------------------------

sample_prediction = logistic_model.predict(
    sample_scaled
)[0]

sample_probability = logistic_model.predict_proba(
    sample_scaled
)[0][1]

print("\n==============================")
print("SAMPLE PASSENGER PREDICTION")
print("==============================")

if sample_prediction == 1:
    print("Prediction: SURVIVED")
else:
    print("Prediction: DID NOT SURVIVE")

print(
    f"Probability of Survival: "
    f"{sample_probability:.2%}"
)

# ============================================================
# MODEL SUMMARY
# ============================================================

# ------------------------------------------------------------
# 34. SUMMARY TABLE
# ------------------------------------------------------------

model_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

print("\n==============================")
print("LOGISTIC REGRESSION RESULTS")
print("==============================")

display(model_results)

# ============================================================
# FINAL CONCLUSION
# ============================================================

print("""
============================================================
CONCLUSION
============================================================

1. Titanic passenger data was loaded and explored.

2. Unnecessary columns were removed.

3. Missing values were handled using median and mode.

4. FamilySize and IsAlone features were created.

5. Categorical variables were encoded.

6. Data was divided into training and testing sets.

7. Features were standardized using StandardScaler.

8. Logistic Regression was trained to predict survival.

9. Model performance was evaluated using:
   - Accuracy
   - Precision
   - Recall
   - F1 Score
   - ROC-AUC

10. A confusion matrix was created to analyze predictions.

11. Logistic Regression coefficients were analyzed to
    understand feature influence.

12. The trained model was also used to predict a sample
    passenger's survival.
============================================================
""")